# Tool Calling

Tool Calling is a method that allows an LLM to use external tools or systems to perform tasks it cannot do by itself.

In [2]:
import ollama 

# TOOL 1: Mock Database for Inventory
def check_inventory(product_name: str):
    """Checks the local database for stock levels of a specific product."""
    inventory_db = {
        "laptop": {"stock": 5, "base_price": 1200},
        "monitor": {"stock": 0, "base_price": 300},
        "keyboard": {"stock": 25, "base_price": 80}
    }
    product = product_name.lower()
    if product in inventory_db:
        return inventory_db[product]
    return {"error": "Product not found"}





In [3]:

# TOOL 2: Business Logic for Discounts
def calculate_loyalty_discount(base_price: float, years_as_customer: int):
    """Calculates a custom discount: 5% per year, capped at 25%."""
    discount_pct = min(years_as_customer * 0.05, 0.25)
    final_price = base_price * (1 - discount_pct)
    return {"final_price": final_price, "discount_applied": f"{discount_pct*100}%"}

In [4]:
# 1. Map the functions so the script can call them by name
available_functions = {
    'check_inventory': check_inventory,
    'calculate_loyalty_discount': calculate_loyalty_discount,
}

In [5]:
# 1. Define the tool schemas for the LLM
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'check_inventory',
            'description': 'Get stock and price for a product',
            'parameters': {
                'type': 'object',
                'properties': {
                    'product_name': {'type': 'string', 'description': 'Name of the product'}
                },
                'required': ['product_name'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'calculate_loyalty_discount',
            'description': 'Calculate final price based on customer loyalty years',
            'parameters': {
                'type': 'object',
                'properties': {
                    'base_price': {'type': 'number', 'description': 'The original price'},
                    'years_as_customer': {'type': 'integer', 'description': 'How many years they have been a member'}
                },
                'required': ['base_price', 'years_as_customer'],
            },
        },
    }
]

In [6]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string',
      'description': 'Name of the product'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on customer loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number',
      'description': 'The original price'},
     'years_as_customer': {'type': 'integer',
      'description': 'How many years they have been a member'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [8]:
# 3. Execution Loop
messages = [{'role': 'user', 'content': 'Do we have laptops in stock? If so, what is the price for a customer who has been with us for 3 years?'}]

# First call: LLM decides which tools to use
response = ollama.chat(model='llama3.2:1b', messages=messages, tools=tools)

In [9]:
response

ChatResponse(model='llama3.2:1b', created_at='2026-02-11T02:20:36.2705501Z', done=True, done_reason='stop', total_duration=9015065600, load_duration=8575183400, prompt_eval_count=248, prompt_eval_duration=70534500, eval_count=19, eval_duration=297982100, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'laptops'}))]), logprobs=None)